# Calculate disease relevance scores

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

#### Library

In [4]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc                    # Clean up
# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data

# dataframes
import pandas as pd

# subprocesses
import subprocess

#### Load ann data

In [2]:
base_dir = str(here("data/disease_relevance"))
files_dir =  os.path.join(base_dir, "files")
anndata_dir = str(here('data/anndata/'))

In [9]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

#### Remove duplicated donors

In [10]:
# Setup -----------------------------------------------------------------------------
sample_key = "ic_id_platform_adjusted_sample"
donor_key  = "ic_id_donor_overall"
dataset_key  = "ic_id_dataset"

# Find donors appearing in multiple datasets
donor_dataset_counts = adata.obs.groupby(donor_key)[dataset_key].nunique()
multi_dataset_donors = donor_dataset_counts[donor_dataset_counts > 1].index

print(f"Found {len(multi_dataset_donors)} donors in multiple datasets")

# Check which multi-dataset donors have 10x available
is_multi = adata.obs[donor_key].isin(multi_dataset_donors)
is_10x = adata.obs['library_prep'].str.contains('10x', case=False)

multi_with_10x = adata.obs[is_multi & is_10x][donor_key].nunique()
multi_without_10x = len(multi_dataset_donors) - multi_with_10x

print(f"  - {multi_with_10x} donors have 10x available (will prioritize)")
print(f"  - {multi_without_10x} donors only in smart-seq (will not keep those)")

# Keep: single-dataset donors + 10x versions of multi-dataset donors
keep = ~is_multi | is_10x
adata = adata[keep].copy()

print(f"Removed {(~keep).sum()} observations")

# Verify filtering worked
remaining_multi = adata.obs.groupby(donor_key)[dataset_key].nunique()
n_remaining_multi = (remaining_multi > 1).sum()
assert n_remaining_multi == 0, f"Error: {n_remaining_multi} donors still appear in multiple datasets!"
print("Filtering successful: all donors now in single dataset")
print("Number of donors: ", len(remaining_multi))

Found 22 donors in multiple datasets
  - 22 donors have 10x available (will prioritize)
  - 0 donors only in smart-seq (will not keep those)
Removed 1118 observations
Filtering successful: all donors now in single dataset
Number of donors:  270


In [13]:
del is_multi, is_10x, multi_with_10x, multi_without_10x, keep, remaining_multi, n_remaining_multi
gc.collect()

6662

#### Create covariate file

In [14]:
cov_df = pd.DataFrame(index=adata.obs_names)
cov_df['donor'] = adata.obs['ic_id_donor_overall'].astype('category').cat.codes
cov_df.to_csv(os.path.join(files_dir, "covariates.tsv"), sep = "\t")

#### Run scDRS per dataset

In [15]:
import os
import subprocess
import anndata as ad
import pandas as pd

# ----------------------------- USER CONFIG -----------------------------------
wd           = "/work/islet_cartography_scrna"
scdrs_bin    = "scdrs"
gs_file      = f"{wd}/data/disease_relevance/files/pops_weight_noweight_small.gs"
cov_file     = f"{wd}/data/disease_relevance/files/covariates.tsv"
out_dir      = f"{wd}/data/disease_relevance/files"
cell_type    = "cell_type"
dataset_col  = "ic_id_dataset"

tmp_dir      = f"{wd}/data/anndata/tmp_split"
os.makedirs(out_dir, exist_ok=True)
os.makedirs(tmp_dir, exist_ok=True)

# ----------------------------- LOAD DATA  ---------------------------------------
if dataset_col not in adata.obs.columns:
    raise ValueError(f"'{dataset_col}' not found in adata.obs columns: {list(adata.obs.columns)}")

datasets = adata.obs[dataset_col].unique().tolist()
print(f"Found {len(datasets)} datasets: {datasets}")

# Load covariates once (assumes first column or index = cell barcode/ID matching adata.obs_names)
cov_df = pd.read_csv(cov_file, sep="\t", index_col=0)

# Traits from the .gs file, needed for the downstream loop
traits = pd.read_csv(gs_file, sep="\t")['TRAIT'].tolist()
print(f"Found {len(traits)} traits: {traits}")

# ----------------------------- LOOP PER DATASET --------------------------------
for dset in datasets:
    print(f"\n{'='*60}")
    print(f"Processing dataset: {dset}")
    print(f"{'='*60}")

    dset_out_dir = os.path.join(out_dir, str(dset))
    os.makedirs(dset_out_dir, exist_ok=True)

    # Subset cells for this dataset
    mask = adata.obs[dataset_col] == dset
    n_cells = mask.sum()
    print(f"  {n_cells} cells in this dataset")

    if n_cells == 0:
        print(f"  Skipping {dset} — no cells found")
        continue

    # Load subset into memory (backed mode -> in-memory subset)
    adata_sub = adata[mask].to_memory()

    # add celltype disease
    adata_sub.obs["cell_type_disease"] = (
        adata_sub.obs[cell_type].astype(str) + "_" + adata_sub.obs["disease_harmonized"].astype(str)
    )

    # Add cell_type_donor
    adata_sub.obs["cell_type_donor"] = (
        adata_sub.obs[cell_type].astype(str) + "_" + adata_sub.obs["ic_id_donor_overall"].astype(str)
    )

    # Subset covariates to matching cells, preserving adata cell order
    cov_sub = cov_df.reindex(adata_sub.obs_names)
    missing_cov = cov_sub.isna().any(axis=1).sum()
    if missing_cov > 0:
        print(f"  Warning: {missing_cov} cells missing covariate rows for {dset}")

    # Write temp h5ad and covariate file for this dataset
    tmp_h5ad = os.path.join(tmp_dir, f"{dset}.h5ad")
    tmp_cov  = os.path.join(tmp_dir, f"{dset}_covariates.tsv")

    adata_sub.write_h5ad(tmp_h5ad)
    cov_sub.to_csv(tmp_cov, sep="\t")

    # ----------------------------- COMPUTE SCORES ------------------------------
    print(f"  Computing scores for {dset}...")
    try:
        result = subprocess.run([
            scdrs_bin, "compute-score",
            "--h5ad-file", tmp_h5ad,
            "--h5ad-species", "human",
            "--gs-file", gs_file,
            "--gs-species", "human",
            "--cov-file", tmp_cov,
            "--adj_prop", cell_type,
            "--flag-filter-data", "True",
            "--flag-raw-count", "False",
            "--out-folder", dset_out_dir
        ], capture_output=True, text=True, check=True)

        print(f"  Finished computing scores for {dset}")
        print(result.stdout)

    except subprocess.CalledProcessError as e:
        print(f"  Command failed for {dset} with exit code {e.returncode}")
        print(f"\n  STDOUT:\n{e.stdout}")
        print(f"\n  STDERR:\n{e.stderr}")
        print(f"\n  Command: {' '.join(str(arg) for arg in e.cmd)}")
        continue

    # ----------------------------- DOWNSTREAM ANALYSIS -------------------------
    group_cols = f"{cell_type},cell_type_disease,cell_type_donor"

    for trait in traits:
        score_file = os.path.join(dset_out_dir, f"{trait}.full_score.gz")
        if not os.path.exists(score_file):
            print(f"    Skipping downstream for {trait} ({dset}) — score file not found: {score_file}")
            continue

        print(f"  Running downstream analysis: {trait} ({group_cols}, {dset})")
        cmd = [
            scdrs_bin, "perform-downstream",
            "--h5ad-file", tmp_h5ad,
            "--score-file", score_file,
            "--out-folder", dset_out_dir + "/",
            "--group-analysis", group_cols,
            "--flag-filter-data", "True",
            "--flag-raw-count", "False"
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"    scDRS downstream failed for {trait} ({group_cols}, {dset})")
            print("    STDOUT:\n", result.stdout)
            print("    STDERR:\n", result.stderr)
            continue
        else:
            print(f"    Success for {trait} ({group_cols}, {dset})")
            
    # cleanup of temp files for this dataset to save disk space
    os.remove(tmp_h5ad)
    os.remove(tmp_cov)

print("\nAll datasets processed.")

Found 24 datasets: ['ic_15', 'ic_3', 'ic_6', 'ic_11', 'ic_8', 'ic_23', 'ic_14', 'ic_16', 'ic_24', 'ic_20', 'ic_21', 'ic_4', 'ic_12', 'ic_2', 'ic_13', 'ic_18', 'ic_1', 'ic_25', 'ic_9', 'ic_22', 'ic_17', 'ic_5', 'ic_19', 'ic_7']
Found 4 traits: ['adult_height', 'type_2_diabetes', 'nw_adult_height', 'nw_type_2_diabetes']

Processing dataset: ic_15
  154776 cells in this dataset
  Computing scores for ic_15...
  Finished computing scores for ic_15
******************************************************************************
* Single-cell disease relevance score (scDRS)
* Version 1.0.4
* Martin Jinye Zhang and Kangcheng Hou
* HSPH / Broad Institute / UCLA
* MIT License
******************************************************************************
Call: scdrs compute-score \
--h5ad-file /work/islet_cartography_scrna/data/anndata/tmp_split/ic_15.h5ad \
--h5ad-species human \
--cov-file /work/islet_cartography_scrna/data/anndata/tmp_split/ic_15_covariates.tsv \
--gs-file /work/islet_cartograp

In [ ]:
# run in terminal first
#cd scDRS
#pip install -e . -q